# Comparison of evaluation of Latxa and Dynamic BPE

In [1]:
import sys
import os
import json

project_root = os.path.expanduser(
    "~/MASTER/WiSe25/Lab Rotation/dynamic-tokenization"
)
sys.path.append(project_root)

In [2]:
# Force offline mode
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# (Optional but recommended)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [3]:
from tokenizations.dynamic_bpe import Dynamic_BPE
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
from zett.utils import get_surface_form_matrix
from collections import Counter
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import torch
from datasets import Dataset

/user/i.elcoroalberdi/u25035/MASTER/WiSe25/Lab Rotation/dynamic-tokenization/venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Load model and tokenizers

In [34]:
# Load hypernetwork tokenizer
hypernet_tokenizer = AutoTokenizer.from_pretrained(
    "benjamin/zett-hypernetwork-Meta-Llama-3-8B-experimental",
    local_files_only=True
)

dynamic_bpe = Dynamic_BPE(
    tokenizer=hypernet_tokenizer,
    tokenizer_boundary="pretokens",
)
print("Hypernetwork + tokenizer + Dynamic BPE ready.")

Hypernetwork + tokenizer + Dynamic BPE ready.


In [22]:
#Load Latxa tokenizer and model
latxa_tokenizer = AutoTokenizer.from_pretrained(
    "HiTZ/latxa-7b-v1.2",
    local_files_only=True
)
model = AutoModelForCausalLM.from_pretrained(
    "HiTZ/latxa-7b-v1.2",
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)
print("Latxa model and tokenizer loaded.")

Loading checkpoint shards: 100%|███████████████████████████| 2/2 [00:07<00:00,  3.65s/it]

Latxa model and tokenizer loaded.


In [23]:
# Quizk sanity check
text = "Kaixo mundua"

print("Latxa tokens:", latxa_tokenizer.tokenize(text))
print("Hypernet tokens:", hypernet_tokenizer.tokenize(text))

Latxa tokens: ['▁Ka', 'ix', 'o', '▁m', 'und', 'ua']
Hypernet tokens: ['ĠKa', 'ixo', 'Ġmund', 'ua']


### Helper functions

In [24]:
def dynamic_tokenize_texts(texts, dynamic_bpe, batch_size=128, max_merges=10, show_progress=True):
    """
    texts: list[str]
    returns: list[list[str]]  (dynamic tokens per text)
    """
    all_tokens = []

    if show_progress:
        iterator = tqdm(range(0, len(texts), batch_size), desc="Dynamic BPE")
    else:
        iterator = range(0, len(texts), batch_size)

    for i in iterator:
        batch_texts = texts[i:i+batch_size]
        batch_examples = [{"text": t} for t in batch_texts]

        dyn_tokens, _, _, _ = dynamic_bpe.tokenize_batch(
            batch_examples=batch_examples,
            max_nr_merges=max_merges,
            mlm=True
        )

        all_tokens.extend(dyn_tokens)

    return all_tokens

In [25]:
def build_batch_tensors(batch_ids, pad_id, device):
    """
    batch_ids: list[list[int]]  (len = 4 choices)
    """
    max_len = max(len(seq) for seq in batch_ids)

    input_ids = torch.full(
        (len(batch_ids), max_len),
        pad_id,
        dtype=torch.long,
        device=device
    )

    attention_mask = torch.zeros_like(input_ids)

    for i, seq in enumerate(batch_ids):
        seq = torch.tensor(seq, dtype=torch.long, device=device)
        input_ids[i, :len(seq)] = seq
        attention_mask[i, :len(seq)] = 1

    return input_ids, attention_mask

In [26]:
def format_prompt(question, candidates):
    return (
        f"Galdera: {question}\n"
        f"A: {candidates[0]}\n"
        f"B: {candidates[1]}\n"
        f"C: {candidates[2]}\n"
        f"D: {candidates[3]}\n"
        f"Erantzuna:"
    )

In [27]:
# Multiple-choice scoring (log-likelihood of last token)
@torch.no_grad()
def score_choices(model, input_ids, attention_mask):
    """
    input_ids: (4, seq_len)
    Returns: tensor of shape (4,) with log-likelihood scores
    """
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    logits = outputs.logits  # (4, seq_len, vocab_size)

    last_token_positions = attention_mask.sum(dim=1) - 1
    scores = []

    for i in range(input_ids.size(0)):
        pos = last_token_positions[i]
        token_id = input_ids[i, pos]
        log_probs = torch.log_softmax(logits[i, pos], dim=-1)
        scores.append(log_probs[token_id])

    return torch.stack(scores)

### Evaluation Dataset 1: EusProficiency

In [28]:
#Load the EusProficiency dataset and prepare evaluation items
arrow_path = (
    "/mnt/vast-react/home/i.elcoroalberdi/u25035/.cache/"
    "huggingface/datasets/HiTZ___eus_proficiency/default/0.0.0/"
    "1b46a247abd611f5435f0b81c8c5fc1320636454/"
    "eus_proficiency-test.arrow"
)
ds = Dataset.from_file(arrow_path)

CHOICES = [" A", " B", " C", " D"]
evaluation_items = []
for item in ds:
    prompt = format_prompt(item["question"], item["candidates"])
    
    choice_texts = [prompt + choice for choice in CHOICES]

    evaluation_items.append({
        "prompt": prompt,
        "choice_texts": choice_texts,   # length 4
        "answer": item["answer"]        # int: 0–3
    })
print(evaluation_items[0])
print("Evaluation items prepared:", len(evaluation_items))

{'prompt': 'Galdera: Bi seme-alaba ditu, ..... ederragoak.\nA: zenbat eta\nB: haiek baino\nC: nola edo hala\nD: zein baino zein\nErantzuna:', 'choice_texts': ['Galdera: Bi seme-alaba ditu, ..... ederragoak.\nA: zenbat eta\nB: haiek baino\nC: nola edo hala\nD: zein baino zein\nErantzuna: A', 'Galdera: Bi seme-alaba ditu, ..... ederragoak.\nA: zenbat eta\nB: haiek baino\nC: nola edo hala\nD: zein baino zein\nErantzuna: B', 'Galdera: Bi seme-alaba ditu, ..... ederragoak.\nA: zenbat eta\nB: haiek baino\nC: nola edo hala\nD: zein baino zein\nErantzuna: C', 'Galdera: Bi seme-alaba ditu, ..... ederragoak.\nA: zenbat eta\nB: haiek baino\nC: nola edo hala\nD: zein baino zein\nErantzuna: D'], 'answer': 3}
Evaluation items prepared: 5169


In [29]:
for item in evaluation_items:
    dynamic_choice_tokens = dynamic_tokenize_texts(
        item["choice_texts"],
        dynamic_bpe,
        batch_size=4,
        show_progress=False  # avoids thousands of nested progress bars
    )
    assert isinstance(dynamic_choice_tokens, list)
    assert isinstance(dynamic_choice_tokens[0], list)
    item["dynamic_tokens"] = dynamic_choice_tokens

print("Dynamic tokenization completed.")

Early exit, 9 out of 10
Early exit, 9 out of 10
Early exit, 8 out of 10
Early exit, 7 out of 10
Early exit, 7 out of 10
Dynamic tokenization completed.


In [49]:
from transformers import PretrainedConfig
import torch

class DummyHypernet:
    def __init__(self, embedding_size):
        class Config(PretrainedConfig):
            hn_surface_maxlen = 50
            hidden_size = embedding_size
        self.config = Config()
        self.device = "cpu"

    def to(self, device):
        self.device = device
        return self

    def __call__(self, surfaces, *args, **kwargs):
        # surfaces: list of strings
        batch_size = len(surfaces) if hasattr(surfaces, "__len__") else 1
        hidden_size = self.config.hidden_size
        pred_in = torch.zeros(batch_size, hidden_size)
        pred_out = torch.zeros(batch_size, hidden_size)
        return pred_in, pred_out, None

embedding_size = model.get_input_embeddings().embedding_dim
dummy_hypernet = DummyHypernet(embedding_size)

augmenter = DynamicAugmenter(
    model=model,
    latxa_tokenizer=latxa_tokenizer,
    hypernet=dummy_hypernet,
    hypernet_tokenizer=hypernet_tokenizer,
    cache_limit=50_000,
    device=device
)

# Manually populate the cache with fake embeddings for missing tokens
for item in evaluation_items:
    for seq in item["dynamic_tokens"]:
        for t in seq:
            if t not in augmenter.vocab and t not in augmenter.cache:
                # Fake embedding
                augmenter.cache[t] = torch.zeros(embedding_size)


In [50]:
# Convert dynamic tokens → token IDs using DynamicAugmenter
all_choice_token_ids = []

for item in tqdm(evaluation_items, desc="Mapping dynamic tokens to IDs"):
    choice_token_ids = augmenter.tokens_to_ids(item["dynamic_tokens"])
    all_choice_token_ids.append(choice_token_ids)

print("Dynamic token → ID conversion completed.")

Mapping dynamic tokens to IDs:   0%|                            | 0/5169 [00:00<?, ?it/s]


IndexError: index 44420 is out of bounds for dimension 0 with size 32010

In [ ]:
# Evaluation loop
pad_id = latxa_tokenizer.pad_token_id or latxa_tokenizer.eos_token_id

correct = 0
total = 0

model.eval()

for item, choice_ids in tqdm(
    zip(evaluation_items, all_choice_token_ids),
    total=len(evaluation_items),
    desc="Evaluating"
):
    input_ids, attention_mask = build_batch_tensors(
        choice_ids,
        pad_id,
        device
    )
    scores = score_choices(model, input_ids, attention_mask)
    predicted = torch.argmax(scores).item()
    if predicted == item["answer"]:
        correct += 1
    total += 1

accuracy = correct / total
print(f"\nFinal accuracy (Dynamic BPE + Hypernet): {accuracy:.4f}")